# Jules Verne Bot — Retraining the RNN on the Shared Split

Companion to `Train_JulesVerne_Transformer.ipynb`. It retrains the **same GRU**
as the original course notebook
(`20_JulesVerneBot_Generating_English_Texts_with_RNN.ipynb`), but on **exactly the
data the Transformers see**, so the two architectures can finally be compared on
held-out text.

The original notebook is kept unchanged as the 2024 course record. Its reported
loss of 0.45 was measured on the training text: the model had no validation set,
so it cannot be compared with the Transformers' held-out losses.

## What changes, and what does not

| | Original notebook | This notebook |
|---|---|---|
| Architecture | Embedding(123→256) → GRU(1024) → Dense(123) | **identical** (4,095,867 parameters) |
| Vocabulary | 123 characters from `books/` | **identical** |
| Window | 120 characters | **identical** |
| Optimizer, batch | Adam 1e-3, batch 128 | **identical** |
| Training text | raw `books/`, Gutenberg license included | `books_clean/` (license removed) |
| Validation | none | 5% from the middle of every book, shared with the Transformers |
| Saved model | last epoch | **best validation epoch** |

> Runtime → Change runtime type → **GPU (T4)** before starting.

## 1. Environment

Colab ships TensorFlow and PyTorch. PyTorch is only needed because the split is
imported from `train_transformer.py`, so there is a single definition of it.

In [ ]:
import sys
import numpy as np
import tensorflow as tf
import keras

print('python     ', sys.version.split()[0])
print('tensorflow ', tf.__version__)
print('keras      ', keras.__version__)
print('GPU        ', tf.config.list_physical_devices('GPU') or 'none -- training will be very slow')

## 2. Get the project

The cell clones the public repository (shallow). If you prefer, upload and unzip
the project instead; the cell finds it either way. If `books_clean/` is missing
it is generated with `clean_corpus.py`, which needs no extra packages.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Mjrovai/Jules_Verne.git'
REQUIRED = ['books', 'vernebot', 'train_transformer.py', 'clean_corpus.py']


def find_project():
    here = Path.cwd().resolve()
    for root in [Path('/content'), here, *here.parents]:
        for candidate in (root / 'Jules_Verne', root):
            if candidate.is_dir() and all((candidate / r).exists() for r in REQUIRED):
                return candidate
    return None


project = find_project()
if project is None and REPO_URL:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/Jules_Verne'], check=True)
    project = find_project()
if project is None:
    raise FileNotFoundError(f'Project not found. It must contain: {REQUIRED}')

os.chdir(project)
sys.path.insert(0, str(project))
if not Path('books_clean').is_dir():
    subprocess.run([sys.executable, 'clean_corpus.py'], check=True)
print('project:', project)

## 3. Vocabulary and the shared split

- **Vocabulary**: rebuilt from the raw `books/`, so it stays at the 123 characters
  the original RNN and every Transformer use.
- **Split**: `split_by_book` holds out a contiguous 5% from the middle of each
  book. It is the same function `train_transformer.py` calls.

In [ ]:
from vernebot.engine import build_vocabulary
from train_transformer import load_books, split_by_book, encode

vocab = build_vocabulary('books')
assert len(vocab) == 123, 'vocabulary changed -- the comparison is no longer fair'

books = load_books('books_clean')
train_segments, val_segments = split_by_book(books, val_fraction=0.05)

print(f'vocab : {len(vocab)} characters')
print(f'books : {len(books)}, {sum(len(t) for _, t in books):,} characters (Gutenberg text removed)')
print(f'train : {sum(len(t) for _, t in train_segments):,} characters in {len(train_segments)} segments')
print(f'val   : {sum(len(t) for _, t in val_segments):,} characters, one slice per book')
print()
print('a validation slice:', repr(val_segments[0][1][:200]))

## 4. Training sequences

As in the original notebook, the text is cut into non-overlapping 121-character
chunks: 120 characters of input and the same 120 shifted by one as target.

The difference is that chunks are cut **inside each segment**, so no sequence
joins the end of one book to the start of another, or reaches across a held-out
slice. The sequences fit in memory, so they are fully shuffled every epoch
instead of through a 10,000-element buffer.

In [ ]:
SEQ_LEN = 120
BATCH_SIZE = 128


def to_sequences(segments):
    chunks = []
    for _, text in segments:
        ids = encode(text, vocab).astype(np.int32)
        n = len(ids) // (SEQ_LEN + 1)
        if n:
            chunks.append(ids[: n * (SEQ_LEN + 1)].reshape(n, SEQ_LEN + 1))
    return np.concatenate(chunks)


train_seq = to_sequences(train_segments)
val_seq = to_sequences(val_segments)

train_ds = (tf.data.Dataset.from_tensor_slices((train_seq[:, :-1], train_seq[:, 1:]))
            .shuffle(len(train_seq), reshuffle_each_iteration=True)
            .batch(BATCH_SIZE, drop_remainder=True)
            .prefetch(tf.data.AUTOTUNE))
val_ds = (tf.data.Dataset.from_tensor_slices((val_seq[:, :-1], val_seq[:, 1:]))
          .batch(BATCH_SIZE)
          .prefetch(tf.data.AUTOTUNE))

print(f'train sequences: {len(train_seq):,}  ({len(train_seq) // BATCH_SIZE} batches per epoch)')
print(f'val sequences  : {len(val_seq):,}')
print(repr(''.join(vocab[i] for i in train_seq[0, :-1])))

## 5. The model

Identical to the course model. The layers are named explicitly, because
`vernebot/engine.py` (the web app and the demo exporter) reads the weights by
these names.

In [ ]:
EMBED_DIM = 256
RNN_UNITS = 1024


def create_model():
    model = keras.Sequential([
        keras.Input(shape=(None,), dtype='int32'),
        keras.layers.Embedding(len(vocab), EMBED_DIM, name='embedding'),
        keras.layers.GRU(RNN_UNITS, return_sequences=True,
                         recurrent_initializer='glorot_uniform', name='gru'),
        keras.layers.Dense(len(vocab), name='dense'),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                  loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
    return model


keras.utils.set_random_seed(1337)
model = create_model()
model.summary()
assert model.count_params() == 4_095_867, 'not the course architecture'

## 6. Train, keeping the best validation epoch

`ModelCheckpoint(save_best_only=True)` writes the model only when validation loss
improves, so the file on disk is always the best epoch, not the last one. The
original run used 30 epochs; that is kept here.

About 30–40 minutes on a T4.

In [ ]:
import time

EPOCHS = 30
OUT = Path('models/rnn-split.keras')
OUT.parent.mkdir(exist_ok=True)

checkpoint = keras.callbacks.ModelCheckpoint(
    OUT, monitor='val_loss', save_best_only=True, verbose=1)

started = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[checkpoint])
train_minutes = (time.time() - started) / 60
print(f'\ntraining time: {train_minutes:.1f} min')

## 7. Evaluate the saved (best) model and write the run record

The record has the same fields as the Transformer runs (`models/tx-*.json`), so
the comparison plot and tables can read all of them the same way.

**On comparability.** Keras averages the loss over all 120 positions of each
window, and `train_transformer.py` does the same over its window. For
`tx-paired-ctx120` (also 120 characters) the numbers are directly comparable.
`tx-paired` sees 256 characters, which is part of what it is being credited for.

In [ ]:
import json
import math

best = keras.models.load_model(OUT)
val_loss = float(best.evaluate(val_ds, verbose=0))
train_loss = float(best.evaluate(train_ds.take(100), verbose=0))

h = history.history
best_epoch = int(np.argmin(h['val_loss']))
steps_per_epoch = len(train_seq) // BATCH_SIZE

record = {
    'config': {'name': 'rnn-split', 'architecture': 'rnn', 'vocab_size': len(vocab),
               'context': SEQ_LEN, 'embed_dim': EMBED_DIM, 'units': RNN_UNITS},
    'parameters': int(best.count_params()),
    'epochs': EPOCHS,
    'steps': EPOCHS * steps_per_epoch,
    'batch_size': BATCH_SIZE,
    'lr': 1e-3,
    'device': 'gpu' if tf.config.list_physical_devices('GPU') else 'cpu',
    'corpus_dir': 'books_clean',
    'vocab_dir': 'books',
    'split': 'book',
    'val_fraction': 0.05,
    'train_characters': int(train_seq.size),
    'val_characters': int(val_seq.size),
    'exported_step': (best_epoch + 1) * steps_per_epoch - 1,
    'exported_epoch': best_epoch + 1,
    'keep_best': True,
    'last_step_val_loss': float(h['val_loss'][-1]),
    'final_train_loss': train_loss,
    'final_val_loss': val_loss,
    'final_val_perplexity': math.exp(val_loss),
    'train_minutes': round(train_minutes, 2),
    'tensorflow_version': tf.__version__,
    'history': [{'epoch': e + 1, 'step': (e + 1) * steps_per_epoch - 1,
                 'train': float(h['loss'][e]), 'val': float(h['val_loss'][e])}
                for e in range(len(h['loss']))],
}
OUT.with_suffix('.json').write_text(json.dumps(record, indent=2))

print(f'best epoch     : {best_epoch + 1} of {EPOCHS}')
print(f'val loss       : {val_loss:.4f}  (perplexity {math.exp(val_loss):.2f})')
print(f'train loss     : {train_loss:.4f}  (100 batches)')
print(f'last-epoch val : {h["val_loss"][-1]:.4f}')
print('wrote', OUT, 'and', OUT.with_suffix('.json'))

## 8. Check that the web app's engine reads the new model

The site does not run TensorFlow: `vernebot/engine.py` reimplements the GRU in
NumPy. Comparing its logits against Keras on validation text proves the exported
file is usable before it is copied anywhere.

In [ ]:
from vernebot.engine import VerneRNN

engine = VerneRNN.load(OUT, vocab=vocab)
window = val_seq[0, :SEQ_LEN]
keras_logits = best(window[None, :], training=False).numpy()[0]

h_state, worst = None, 0.0
for t, idx in enumerate(window):
    np_logits, h_state = engine.logits([int(idx)], h_state)
    worst = max(worst, float(np.abs(np_logits - keras_logits[t]).max()))

print(f'max |logit difference| over {SEQ_LEN} positions: {worst:.2e}')
assert worst < 1e-3, 'the NumPy engine disagrees with Keras'

## 9. Loss curves, next to the Transformers

Each run's validation loss against the number of characters it had trained on
(steps × batch × window), because the RNN and the Transformers use different
batch sizes and windows. Transformer records are read from `models/` when they
are there; retrain them with the new split first, or the curves are not
comparable.

In [ ]:
import matplotlib.pyplot as plt

runs = {'RNN (rnn-split)': record}
for label, name in (('Transformer ctx120', 'tx-paired-ctx120'),
                    ('Transformer ctx256', 'tx-paired'),
                    ('Transformer large', 'tx-large')):
    p = Path(f'models/{name}.json')
    if p.exists():
        run = json.loads(p.read_text())
        if run.get('split') != 'book':
            print(f'skipping {name}: trained on the old split')
            continue
        runs[label] = run

plt.figure(figsize=(11, 5))
for label, run in runs.items():
    chars = run['batch_size'] * run['config']['context']
    xs = [(r['step'] + 1) * chars / 1e6 for r in run['history']]
    plt.plot(xs, [r['val'] for r in run['history']], marker='o', ms=3, label=label)
plt.xlabel('training characters seen (millions)')
plt.ylabel('validation loss (nats/character)')
plt.title('Held-out loss on the shared per-book split')
plt.grid(alpha=.3); plt.legend(); plt.show()

for label, run in runs.items():
    print(f"{label:22s} params {run['parameters']:>11,}  val {run['final_val_loss']:.4f}  "
          f"ppl {run['final_val_perplexity']:.2f}")

## 10. A sample

Generated with the NumPy engine, the same code path as the web app.

In [ ]:
rng = np.random.default_rng(42)
for temperature in (0.5, 0.7, 1.0):
    text = ''.join(engine.generate('THE FLYING SUBMARINE', num_generate=500,
                                   temperature=temperature, rng=rng))
    print(f'--- temperature {temperature} ---')
    print(text, '\n')

## 11. Download

Put both files in the project's `models/` folder. The web app discovers any
`*.keras` there as an RNN.

In [ ]:
try:
    from google.colab import files
    files.download(str(OUT))
    files.download(str(OUT.with_suffix('.json')))
except ImportError:
    print('Not running in Colab; the files are in', OUT.parent.resolve())